# Random Forest

Il Random Forest è un algoritmo di machine learning supervisionato basato su una collezionei alberi decisionali indipendenti, combinati per migliorare la accuratezza e la robustezza rispetto a singoli alberi.

In [55]:
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from pathlib import Path
import warnings
# Nascondo i warning
warnings.filterwarnings('ignore')

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Lavoro su singoli fold

In [56]:
def fit_single_fold(train_idx, test_idx, features, target, groups, n_estimators, max_depth, min_samples_split, min_samples_leaf, csv_name, fold):
    # Per debug: stampo indici train/test e gruppi
    """print("?"*50 + "\nDebug fold " + str(fold) + "\n" + "?"*50)
    print(f"Fold {fold} - File: {csv_name}")        
    print(f"  Train indice (prime 10): {train_idx[:10]}")
    print(f"  Test indice (prime 10): {test_idx[:10]}")
    print(f"  Train gruppo (Patient IDs): {groups.iloc[train_idx].unique()}")
    print(f"  Test gruppo (Patient IDs): {groups.iloc[test_idx].unique()}")
    print("?"*50 + "\n")"""

    X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
    y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42,
        n_jobs=-1  # parallelizza training interno agli alberi
    )

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    score = f1_score(y_test, y_pred, average="micro")
    
    return score

# Training

In [57]:
def training(file_path, csv_name):

    # Legge il dataset
    df = pd.read_csv(file_path)

    # Definisco le colonne target
    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]']
    
    # Vado a rimuovere le lesioni (righe) non valide
    df_validi = df.dropna(subset=original_target_list).copy()

    # Trasformo tutto in valori binari per "facilitare" il lavoro
    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)  
    
    # Lista finale delle colonne target binarizzate che verranno usate per l'addestramento
    final_target_list = ['PR_class', 'ER_class', 'KI67_class']
    
    """
     Preparo le feature (X) e i target (y) per il modello
    """
    # Definisco tutte le colonne da rimuovere per ottenere solo le feature radiomiche
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    # 'target' contiene le 3 colonne da usare
    target = df_validi[final_target_list]
    
    # 'groups' contiene l'ID del paziente per ogni lesione.
    # Mi serve per fare la cross-validation a gruppo
    groups = df_validi['Patient ID']

    # Riempip a Nan se è rimasto vuoto
    features = features.fillna(features.mean())


    # Istanzio il classificatore RandomForest con parametri standard
    # random_state=42 garantisce che i risultati siano riproducibili
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    

    # Definisco gli iperparametri
    iperparametri = {
        'n_estimators': [100, 200],     # numero di stimatori costruiti
        'max_depth': [10, 20, None],    # profondità massima, 10 e 20 = profondità che "limitano" overfitting e None = depth libera
        'min_samples_split': [2, 5],    # numero minimo di campioni necessari per fare la divisione sui campioni in un nodo, 2 default e 5 lo suo per rendere l'albero generico
        'min_samples_leaf': [1, 2]      # numero minimo di campioni necessari in una foglia, 1 = permissivo e 2 = prevengo che le foglie siano piccine
    }


    # Lista vuolta per collezionare i risultati
    results = []

    cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)

    for n_estimators in iperparametri['n_estimators']:
        for max_depth in iperparametri['max_depth']:
            for min_samples_split in iperparametri['min_samples_split']:
                for min_samples_leaf in iperparametri['min_samples_leaf']:
                    # Parallelizzo fold
                    fold_scores = Parallel(n_jobs=-1)(
                        delayed(fit_single_fold)(train_idx, test_idx, features, target, groups,
                                                 n_estimators, max_depth, min_samples_split, min_samples_leaf,
                                                 csv_name, fold)
                        for fold, (train_idx, test_idx) in enumerate(cv.split(features, target, groups))
                    )
                    
                    # calcolo media e deviazione standard degli score su tutte le fold
                    mean_score = np.mean(fold_scores)
                    std_score = np.std(fold_scores)

                    # registro i risultati per la combinazione di parametri corrente
                    results.append({
                        'n_estimators': n_estimators,
                        'max_depth': max_depth,
                        'min_samples_split': min_samples_split,
                        'min_samples_leaf': min_samples_leaf,
                        'mean_score': mean_score,
                        'std_score': std_score,
                        'fold_scores': fold_scores
                    })


    return results


# Lettura dei file

In [58]:
results_per_dataset = {}

for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Stampo i risultati per ogni dataset
for name, metrics_list in results_per_dataset.items():
    
    # Stampo solo i migliori
    best_result = max(metrics_list, key=lambda x: x['mean_score'])
    print(f"\nDataset: {name}:")
    print(f"  n_estimators: {best_result['n_estimators']}")
    print(f"  max_depth: {best_result['max_depth']}")
    print(f"  min_samples_split: {best_result['min_samples_split']}")
    print(f"  min_samples_leaf: {best_result['min_samples_leaf']}")
    print(f"  Mean F1-score: {best_result['mean_score']:.3f} ± {best_result['std_score']:.3f}\n")


# va dentro al for sopra



Dataset: t2_medsam:
  n_estimators: 200
  max_depth: 10
  min_samples_split: 2
  min_samples_leaf: 1
  Mean F1-score: 0.757 ± 0.065


Dataset: t2_preprocessed:
  n_estimators: 200
  max_depth: 10
  min_samples_split: 2
  min_samples_leaf: 2
  Mean F1-score: 0.758 ± 0.083


Dataset: t2_original:
  n_estimators: 100
  max_depth: 10
  min_samples_split: 5
  min_samples_leaf: 2
  Mean F1-score: 0.768 ± 0.092


Dataset: medsam_dynamic:
  n_estimators: 100
  max_depth: 10
  min_samples_split: 5
  min_samples_leaf: 2
  Mean F1-score: 0.747 ± 0.066


Dataset: preprocessed_dynamic:
  n_estimators: 200
  max_depth: 10
  min_samples_split: 2
  min_samples_leaf: 2
  Mean F1-score: 0.739 ± 0.050


Dataset: original_dynamic:
  n_estimators: 200
  max_depth: 10
  min_samples_split: 5
  min_samples_leaf: 1
  Mean F1-score: 0.750 ± 0.028



"""
print(f"\nNome CSV: {name}")
for res in metrics_list:
    scores_per_fold = res['fold_scores']
    formatted_scores = [f'{s:.3f}' for s in scores_per_fold]
    print(f"Params: n_estimators={res['n_estimators']}, max_depth={res['max_depth']}, "
            f"min_samples_split={res['min_samples_split']}, min_samples_leaf={res['min_samples_leaf']}")
    #print(f"Scores per fold: {formatted_scores}")
    print(f"Media e Dev. Std.: {res['mean_score']:.3f} ± {res['std_score']:.3f}\n")
"""